In [34]:
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI, AsyncOpenAI
from llm_utils import llm_call, llm_call_async
from typing import List
import asyncio
from lib.tools import parse_md_json

async_client = AsyncOpenAI()
sync_client = OpenAI()

# 웹 검색 기능을 포함한 LLM 호출 함수 선언
async def llm_search_async(prompt: str, model: str = "gpt-4.1") -> str:
    response = await async_client.responses.create(
        model=model,
        input=prompt,
        tools=[{"type": "web_search"}],
    )
    return response.output_text

async def run_llm_parallel(prompt_details):
    tasks = [
        llm_search_async(item['user_prompt'], item['model'])
        for item in prompt_details
    ]
    responses = await asyncio.gather(*tasks)
    return responses

In [59]:
test_res = llm_call("너 뭐야", model="gpt-4o")
print(f' ==> [Line 1]: \033[38;2;216;234;16m[test_res]\033[0m({type(test_res).__name__}) = \033[38;2;19;48;199m{test_res}\033[0m')


 ==> [Line 1]: [test_res](str) = 저는 인공지능 기반의 대화형 모델이에요. 궁금한 점이나 도움이 필요한 게 있으면 말씀해 주세요.


# 자소서 질문 생성기

## Phase 1 · JD 분석

### 1-1 워커 실행 함수 및 모델 라우팅 준비

In [35]:
import json
from lib.phase1 import run_jd_workers
from llm_utils.utils import llm_call_async

# 1. 모델명에 따라 일반 LLM 호출과 검색(Search) LLM 호출을 라우팅하는 래퍼 함수
async def notebook_llm_async_fn(prompt: str, model: str) -> str:
    print(f"[{model}] 호출 중...")
    if model == "gpt-4.1": 
        # 첫 번째 셀에 정의된 llm_search_async 사용
        return await llm_search_async(prompt, model=model)
    else:
        # 워커 기본 모델은 llm_call_async 사용
        return await llm_call_async(prompt, model=model)


### 1-2 JD 샘플 텍스트 로드

In [36]:
# 2. JD 샘플 텍스트 로드
with open('JD-sample.md', 'r', encoding='utf-8') as f:
    jd_text = f.read()

### 1-3 Phase 1 병렬 실행 및 결과 확인

In [37]:
# 3. Phase 1 파이프라인 병렬 실행
phase1_results = await run_jd_workers(jd_text, notebook_llm_async_fn)

[gpt-4o] 호출 중...
[gpt-4o] 호출 중...
[gpt-4o] 호출 중...
[gpt-4.1] 호출 중...
[gpt-4.1] 호출 중...
gpt-4o 완료
gpt-4o 완료
gpt-4o 완료


In [38]:
# 4. 결과 확인
print("\n" + "="*50)
print("✅ Phase 1 분석 결과 (JSON Wrapper)")
print("="*50)
for res in phase1_results:
    print(f"\n▶ Agent ID: {res['agent_id'].upper()}")
    print(json.dumps(res, ensure_ascii=False, indent=2))


✅ Phase 1 분석 결과 (JSON Wrapper)

▶ Agent ID: ROLE
{
  "company_name": "삼성전자 DS부문 AI센터",
  "job_title": "SW개발",
  "agent_id": "role",
  "payload": {
    "roles": [
      {
        "role_name": "반도체 도메인 특화 AI 모델, Agent 시스템/서비스, Platform 연구 및 개발",
        "required_skills": [
          "AI 모델링",
          "플랫폼 개발",
          "반도체 도메인 지식"
        ],
        "question_type": "직무적합"
      },
      {
        "role_name": "AI Agent 아키텍처 연구 및 설계",
        "required_skills": [
          "AI 아키텍처 설계",
          "데이터 기반 설계"
        ],
        "question_type": "의사결정"
      },
      {
        "role_name": "LLM 기반 추론 구조 및 멀티 Agent 협업 시스템 개발",
        "required_skills": [
          "LLM",
          "멀티 Agent 시스템"
        ],
        "question_type": "기술깊이"
      },
      {
        "role_name": "반도체 분석 Workflow 자동화 Agent 설계 및 고도화",
        "required_skills": [
          "자동화 시스템",
          "Workflow 설계"
        ],
        "question_type": "의사결정"
      },
      {
        "role_name": "Tool 연계, RAG 구조 및 

## Phase 2 · 자소서 분석 - 프롬프트 체이닝

### 2-1 자기소개서 각 항목 분류

In [39]:
import importlib
import lib.phase2 as phase2
from lib.tools import parse_md_json

importlib.reload(phase2)
from lib.phase2 import classify_personal_statement_sections

file_path = './essay-sample.md'
with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

personal_statement_res = classify_personal_statement_sections(content)
personal_statement_list = parse_md_json(personal_statement_res)
if isinstance(personal_statement_list, dict):
    personal_statement_list = [personal_statement_list]

print(f' ==> [Line 11]: \033[38;2;85;130;182m[personal_statement_list]\033[0m({type(personal_statement_list).__name__}) = \n\033[38;2;248;199;101m{personal_statement_list}\033[0m')

 ==> [Line 11]: [personal_statement_list](list) = 
[{'question': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오.', 'answer': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다] 삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다. 입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.'}, {'question': '본인의 성장과정을 간략히 기술하되 현재의 자신에게 가장 큰 영향을 끼친 사건, 인물 등을 포함하여 기술하시기 바랍니다.', 'answer': "[통제 가능한 루틴으로 불확실성을 극복하다] 저의 성장을 이끈 핵심 동력은 명확한 규칙을 세우고 이를 매일 지켜내는 '실행력'입니다. 대학 시절 4.43/4.5의 최종 학점을

### 2-2 각 JD와 자기소개서 항목을 비교해서 매칭 & 분석

In [40]:
from pprint import pprint
from lib.phase2 import analyze_and_match_essay

sample_jd_list = [item for item in phase1_results if item["agent_id"] == "role"]
if not sample_jd_list:
    raise ValueError("phase1_results에서 agent_id='role' 결과를 찾지 못했습니다.")

target_jd = sample_jd_list[0]

print("=" * 50)
print("=== [STEP 1] analyze_and_match_essay ===")
print("=" * 50)
analyzed_result = analyze_and_match_essay(
    js_list=target_jd,
    personal_statement_list=personal_statement_list,
    client=sync_client,
    model="gpt-4o-mini",
)
pprint(analyzed_result)

=== [STEP 1] analyze_and_match_essay ===
{'items': [{'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다] 삼성전자 '
                           'DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 '
                           '플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 '
                           '결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 '
                           '삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 '
                           '협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 '
                           '실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 '
                           '지원했습니다. 입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent '
                           '백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. '
                           '첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 '
                           '데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 '
                           '결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 '
  

### 2-3 build_question_context

In [41]:
from lib.phase2 import build_question_context

print("\n" + "=" * 50)
print("=== [STEP 2] build_question_context ===")
print("=" * 50)
context_result = build_question_context(
    analyzed_result=analyzed_result,
    client=sync_client,
    model="gpt-4o-mini",
)
print(context_result)


=== [STEP 2] build_question_context ===
{'target': {'company': '삼성전자', 'division': 'DS부문 AI센터', 'role': 'SW개발'}, 'items': [{'question_id': 'Q1', 'question_text': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오.', 'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다] 삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다. 입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.', 'item_type': 'motivation_vision', 'question_intent': ['회사에 대한 이해', '장기적 비전 제시', '문제 해결 능력

### 2-4 결과물 출력

In [42]:
from lib.phase2 import aggregate_phase2_results

print("\n" + "=" * 50)
print("=== [STEP 3] aggregate_phase2_results ===")
print("=" * 50)
final_output = aggregate_phase2_results(
    js_list=target_jd,
    contextualized_result=context_result,
    client=sync_client,
    model="gpt-4o-mini",
)

phase2_result = final_output
pprint(final_output)


=== [STEP 3] aggregate_phase2_results ===
{'global_context': {'global_risks': ['설계 및 구현의 과장 가능성',
                                     '실제 성과와의 괴리',
                                     '구체적인 기술적 세부 사항의 부족',
                                     '과장된 성과 설명 가능성',
                                     '개별 경험의 일반화에 대한 우려',
                                     '기술 설명 부족으로 인한 오해의 가능성',
                                     '하드웨어 의존성 과장'],
                    'missing_jd_ids': ['R2',
                                       'R3',
                                       'R4',
                                       'R7',
                                       'R8',
                                       'R9',
                                       'R11',
                                       'R14',
                                       'R16'],
                    'priority_question_topics': [{'priority': 1,
                                                  'reason': 'AI와 반도체 분야에서의 데이터 '
         

## Phase 3 · 질문 산출

- 오케스트레이터 프롬프트 생성

In [43]:
def get_orchestrator_prompt(phase2_result):
    target = phase2_result.get("target", {})
    company = target.get("company", "해당 기업")
    role = target.get("role", "지원 직무")

    return f"""
시스템 역할: 
너는 {company} {role} 채용 전략가이자 면접 설계자야. 
전달받은 자소서 분석 데이터를 바탕으로, 이번 면접에서 검증이 가장 시급한 '핵심 카테고리' 5개를 선정하고 각 카테고리별 공략 지점을 정해줘.

분석 데이터 맥락:
- 타겟: {company} / {role}
- 전역 맥락(Global Context): {phase2_result.get('global_context', {})}

수행 태스크:
입력된 데이터를 분석하여 아래 5개 카테고리의 우선순위를 정하고, **워커(Worker) 면접관이 해당 카테고리에서 어떤 부분을 중점적으로 파고들어야 할지(Reason)**를 명시해줘.

[카테고리 후보]
1. 경험검증 / 2. 기술깊이 / 3. 직무적합성 / 4. 기업 핏 문화정합 / 5. 꼬리,인성

출력 제약:
- 반드시 아래 JSON 배열 형식으로만 출력할 것.
- `reason`은 해당 카테고리에서 검증해야 할 구체적인 소재나 의도를 1문장으로 작성할 것.

[출력 형식]
[
    {{"category": "카테고리명", "priority": 1, "reason": "이 에피소드의 수치적 진위와 실제 기여도 확인 필요"}},
    {{"category": "카테고리명", "priority": 2, "reason": "사용한 A 기술의 아키텍처 이해도와 트레이드오프 경험 검증"}},
    ... (총 5개)
]

전체 분석 결과 데이터:
{phase2_result}
"""

- 오케스트레이터 프롬프트 JSON파싱

In [44]:
import json

# 오케스트레이터를 위한 프롬포트 생성
orchestrator_prompt = get_orchestrator_prompt(phase2_result)

# 오케스트레이터가 질문을 복제
orchestrator_response = llm_call(orchestrator_prompt, model='gpt-4o')

# 잡다한 텍스트 없애기
clean_orchestrator_response = orchestrator_response.replace("```json", '').replace("```", "").strip()

subtask_list = json.loads(clean_orchestrator_response)
subtask_list

[{'category': '경험검증', 'priority': 1, 'reason': '이 에피소드의 수치적 진위와 실제 기여도 확인 필요'},
 {'category': '기술깊이',
  'priority': 2,
  'reason': '사용한 A 기술의 아키텍처 이해도와 트레이드오프 경험 검증'},
 {'category': '직무적합성',
  'priority': 3,
  'reason': 'AI 데이터 파이프라인 구축 경험의 실제 활용성 확인 필요'},
 {'category': '기업 핏 문화정합',
  'priority': 4,
  'reason': '삼성전자의 비전과 핵심 가치에 대한 지원자의 이해도 평가'},
 {'category': '꼬리,인성', 'priority': 5, 'reason': '문제 상황에서의 인내와 적응력 평가가 필요'}]

- 워커 프롬프트 생성

In [45]:
def get_worker_prompt(subtask_dict, phase2_result):
    """
    subtask_dict: 오케스트레이터가 생성한 개별 객체 ({"category": "...", "reason": "..."})
    phase2_result: 전체 자소서 분석 결과 JSON
    """
    # 기본 정보 추출
    target = phase2_result.get("target", {})
    company = target.get("company", "해당 기업")
    role = target.get("role", "지원 직무")
    
    # 오케스트레이터가 넘겨준 세부 전략 추출
    category = subtask_dict.get("category", "공통 역량")
    reason = subtask_dict.get("reason", "전반적인 실무 역량 검증")

    return f"""
시스템 역할: 
너는 {company} {role} 면접의 '기술 압박 면접관'이야. 

[검증 전략]
- 집중 검증 카테고리: {category}
- **검증 의도 및 이유: {reason}**

수행 지침:
1. (데이터 추출) 위 **검증 의도({reason})**를 최우선으로 반영하여, 자소서 분석 데이터(phase2_result)의 `items` 중 가장 적합한 소재를 선정해.
2. (질문 생성) 해당 소재에 대해 실무 역량을 바닥까지 파헤치는 **심층 질문을 반드시 4개** 생성해.
3. (단계적 압박) 
   - 질문 1: 경험의 구체적 사실 관계 및 본인의 기여도 확인
   - 질문 2: 사용한 기술/방법론의 선택 근거와 작동 원리(Deep Dive)
   - 질문 3: 예상치 못한 변수나 실패 상황에서의 트레이드오프 대처 능력
   - 질문 4: {company}의 비즈니스나 최신 기술 트렌드와 결합된 실무 적용 질문
4. (형식 제약) 다른 설명 없이 **오직 JSON 배열 형식의 질문 문장만** 출력해.

[출력 형식]
[
    "질문 문장 1",
    "질문 문장 2",
    "질문 문장 3",
    "질문 문장 4"
]

자소서 분석 데이터:
{phase2_result}
"""

- 모든 하위 질문에 대해 워커 프롬프트를 생성

In [46]:
worker_prompt_details = [
    {
        "user_prompt": get_worker_prompt(subtask, phase2_result),
        "model": "gpt-4o" # 모델명 오타 수정 (gpt-4.1 -> gpt-4o 등 실제 모델명으로)
    }
    for subtask in subtask_list
]

- worker_responses 변수에 워커 응답 담기

In [47]:
worker_responses = await run_llm_parallel(worker_prompt_details)

- 어그리게이터 프롬프트 생성

In [48]:
def get_aggregator_prompt(worker_responses, phase2_result):
    """
    worker_responses: 모든 워커가 생성한 20개의 질문 리스트 (카테고리 정보 포함)
    phase2_result: 타겟 기업/직무 정보
    """
    target = phase2_result.get("target", {})
    company = target.get("company", "해당 기업")
    role = target.get("role", "지원 직무")

    return f"""
시스템 역할: 
{company} {role} 면접 대비를 위한 '실전 질문 리스트' 최종 통합해야해.
분야별로 조사된 질문들을 지원자가 연습하기 좋게 **오직 질문만** 출력해.

분석 대상: {company} / {role}

통합 지침:
1. (누락 없는 나열) 워커들이 전달한 총 20개의 질문을 하나도 빠짐없이 모두 포함해.
2. (형식 제한) 부연 설명, 답변 가이드, 질문 의도는 모두 삭제해. 오직 **질문 문장**만 리스트 형태로 출력해.
3. 결과는 JSON형태로 출력해줘.

[워커들의 전체 데이터]
{worker_responses}
"""

- 워커 응답 합치기

In [49]:
aggregator_prompt = get_aggregator_prompt(worker_responses, phase2_result)
final_response = llm_call(aggregator_prompt, model="gpt-4.1")
print(final_response)

```json
[
    "당신이 AI 서빙 플랫폼의 성능을 향상시킨 경험에 대해 자세히 설명해주십시오. 구체적으로 어떤 문제를 해결했으며, 당신의 기여 역할은 무엇이었나요?",
    "AI 서빙 플랫폼 성능을 향상시키는 데 사용한 기술이나 방법론의 선택 근거와 그 작동 원리를 설명해주십시오.",
    "프로젝트 진행 중에 예상치 못한 변수가 발생했다면 어떻게 대처했는지, 그 과정에서의 트레이드오프 결정은 무엇이었나요?",
    "삼성전자의 최신 AI 기술 트렌드와 어떻게 접목하여 AI 서빙 플랫폼의 성능을 더욱 향상시킬 계획인가요?",
    "당신이 참여했던 AI 서빙 플랫폼의 성능 향상 프로젝트에서 구체적으로 어떤 역할을 수행했는지 설명해 주세요.",
    "해당 프로젝트에서 사용한 AI 플랫폼의 아키텍처를 선택한 이유와 그 기술이 어떻게 작동하는지 설명해 주세요.",
    "프로젝트 진행 중 응답 지연이나 성능 저하 문제가 발생했을 때, 이를 해결하기 위해 어떤 트레이드오프를 선택했는지 사례를 들어 설명해 주세요.",
    "삼성전자의 반도체 제조 공정에 현재의 AI 플랫폼 기술을 접목한다면, 어떤 방식으로 실무 적용이 가능할지 구체적으로 설명해 주세요.",
    "당신이 AI 데이터 파이프라인 구축을 주도한 프로젝트에 대해 자세히 설명해주시고, 프로젝트에서 당신의 구체적인 역할과 기여도가 무엇이었는지 말씀해 주시겠어요?",
    "해당 프로젝트에서 사용한 데이터 파이프라인 기술 스택을 선택한 이유와 각 기술의 작동 원리에 대해 깊이 설명해 주세요.",
    "프로젝트 진행 중 예상치 못한 문제나 장애가 발생했을 때 어떻게 대처하셨나요? 이를 해결하는 과정에서 고려했던 트레이드오프는 무엇이었는지요?",
    "삼성전자의 반도체 제조 공정에 당신이 구축한 AI 데이터 파이프라인을 적용한다면, 어떤 방식으로 성능을 최적화하려고 하겠습니까?",
    "AI 서빙 플랫폼의 성능 향상 경험에 대해 설명하고, 구체적으로 본인의 기여가 무엇이었는

- 합친 응답 JSON 파싱

In [50]:
clean_final_response = final_response.replace("```json", '').replace("```", "").strip()
phase3_result = json.loads(clean_final_response)
phase3_result

['당신이 AI 서빙 플랫폼의 성능을 향상시킨 경험에 대해 자세히 설명해주십시오. 구체적으로 어떤 문제를 해결했으며, 당신의 기여 역할은 무엇이었나요?',
 'AI 서빙 플랫폼 성능을 향상시키는 데 사용한 기술이나 방법론의 선택 근거와 그 작동 원리를 설명해주십시오.',
 '프로젝트 진행 중에 예상치 못한 변수가 발생했다면 어떻게 대처했는지, 그 과정에서의 트레이드오프 결정은 무엇이었나요?',
 '삼성전자의 최신 AI 기술 트렌드와 어떻게 접목하여 AI 서빙 플랫폼의 성능을 더욱 향상시킬 계획인가요?',
 '당신이 참여했던 AI 서빙 플랫폼의 성능 향상 프로젝트에서 구체적으로 어떤 역할을 수행했는지 설명해 주세요.',
 '해당 프로젝트에서 사용한 AI 플랫폼의 아키텍처를 선택한 이유와 그 기술이 어떻게 작동하는지 설명해 주세요.',
 '프로젝트 진행 중 응답 지연이나 성능 저하 문제가 발생했을 때, 이를 해결하기 위해 어떤 트레이드오프를 선택했는지 사례를 들어 설명해 주세요.',
 '삼성전자의 반도체 제조 공정에 현재의 AI 플랫폼 기술을 접목한다면, 어떤 방식으로 실무 적용이 가능할지 구체적으로 설명해 주세요.',
 '당신이 AI 데이터 파이프라인 구축을 주도한 프로젝트에 대해 자세히 설명해주시고, 프로젝트에서 당신의 구체적인 역할과 기여도가 무엇이었는지 말씀해 주시겠어요?',
 '해당 프로젝트에서 사용한 데이터 파이프라인 기술 스택을 선택한 이유와 각 기술의 작동 원리에 대해 깊이 설명해 주세요.',
 '프로젝트 진행 중 예상치 못한 문제나 장애가 발생했을 때 어떻게 대처하셨나요? 이를 해결하는 과정에서 고려했던 트레이드오프는 무엇이었는지요?',
 '삼성전자의 반도체 제조 공정에 당신이 구축한 AI 데이터 파이프라인을 적용한다면, 어떤 방식으로 성능을 최적화하려고 하겠습니까?',
 'AI 서빙 플랫폼의 성능 향상 경험에 대해 설명하고, 구체적으로 본인의 기여가 무엇이었는지 말씀해 주세요.',
 '당시 사용했던 기술이나 방법론 중 어떤 것을 선택했으며, 그 

## Phase 4 · 평가-최적화 루프

In [51]:
from lib.phase4 import optimize_questions_with_retries, get_passed_final_questions

# phase3_result를 그대로 사용
phase4_questions = phase3_result

# phase2_result에서 role(직무) 정보 추출
job_category = phase2_result.get('target', {}).get('role', '백엔드개발자')

phase4_results = optimize_questions_with_retries(
    phase4_questions,
    job_category,
    max_retries=3,
)

phase4_pass_questions = get_passed_final_questions(phase4_results)

phase4_results, phase4_pass_questions

([{'original_question': '당신이 AI 서빙 플랫폼의 성능을 향상시킨 경험에 대해 자세히 설명해주십시오. 구체적으로 어떤 문제를 해결했으며, 당신의 기여 역할은 무엇이었나요?',
   'initial_status': 'PASS',
   'initial_evaluation': '평가 결과: PASS\n\n문제점 및 개선 방향: 없음\n\n이 질문은 AI 서빙 플랫폼의 성능 향상에 대한 경험을 묻고 있으며, 면접자가 문제 해결 능력과 기여 역할을 구체적으로 설명할 기회를 제공합니다. 다음 평가 기준을 모두 충족합니다:\n\n1. 꼬리질문으로 이어질 수 있는가: 질문은 구체적이며, 또 다른 관련 질문(예: 어떤 기술 스택을 사용했나요? 결과는 어떻게 측정했나요?)으로 이어질 수 있습니다.\n2. 질문이 구체적인가: AI 서빙 플랫폼의 성능 향상이라는 특정 주제를 다루고 있어 명확하고 구체적입니다.\n3. JD에서 요구하는 역량과 연결되는가: AI, 성능 최적화 및 문제 해결 관련 역량을 평가할 수 있어 JD에서 요구하는 조건과 잘 연결됩니다.\n4. 자기소개서 기반으로 답할 수 있는가: 면접자가 자신의 경험을 바탕으로 구체적으로 설명할 수 있는 기회를 제공합니다.\n5. 실제 면접관이 할 법한 질문인가: 실제 SW 개발 관련 면접에서 자주 등장할 수 있는 질문 유형입니다.\n\n따라서 면접 질문은 모두 PASS 기준을 충족하고 있습니다.',
   'attempts': [],
   'final_question': '당신이 AI 서빙 플랫폼의 성능을 향상시킨 경험에 대해 자세히 설명해주십시오. 구체적으로 어떤 문제를 해결했으며, 당신의 기여 역할은 무엇이었나요?',
   'final_status': 'PASS',
   'final_evaluation': '평가 결과: PASS\n\n문제점 및 개선 방향: 없음\n\n이 질문은 AI 서빙 플랫폼의 성능 향상에 대한 경험을 묻고 있으며, 면접자가 문제 해결 능력과 기여 역할을 구체적으로 설명할 기회를 제공

- 원본 질문 평가
- PASS면 바로 종료
- FAIL이면 improve_question()으로 질문 개선
- 개선 질문을 다시 phase4()로 평가
- 또 FAIL이면 max_retries 만큼 반복
- 중간에 PASS가 나오면 즉시 종료
- 끝까지 FAIL이면 마지막 개선 질문이 final_question이 됨

## Phase 5 · 최종 출력

In [52]:
import importlib
import lib.phase5 as phase5_module
importlib.reload(phase5_module)
from lib.phase5 import phase5, format_phase5_output

# 1. Phase 4를 통과한 최종 질문 리스트를 사용
test_questions = phase4_pass_questions

# 2. 직무 카테고리 설정 (Phase 4에서 이미 정의된 job_category 사용)
# job_category = job_category

print(f"🚀 [Phase 5 실행] '{job_category}' 직무의 면접 질문 분석 중...\n")

try:
    # 3. Phase 5 실행 (분류, 채점, 정렬, 요약)
    result_data = phase5(test_questions, job_category)
    
    # 4. 결과 포맷팅 및 출력
    report_text = format_phase5_output(result_data)
    
    print("=" * 50)
    print(report_text)
    print("=" * 50)
    
except Exception as e:
    print(f"❌ 실행 중 오류 발생: {e}")


🚀 [Phase 5 실행] 'SW개발' 직무의 면접 질문 분석 중...

🏆 직무별 핵심 예상 질문 리포트

■ 2) 프로젝트 경험 검증
  1위. 프로젝트 진행 중에 예상치 못한 변수가 발생했다면 어떻게 대처했는지, 그 과정에서의 트레이드오프 결정은 무엇이었나요?
  2위. 당신이 AI 데이터 파이프라인 구축을 주도한 프로젝트에 대해 자세히 설명해주시고, 프로젝트에서 당신의 구체적인 역할과 기여도가 무엇이었는지 말씀해 주시겠어요?
  3위. 프로젝트 진행 중 예상치 못한 문제나 장애가 발생했을 때 어떻게 대처하셨나요? 이를 해결하는 과정에서 고려했던 트레이드오프는 무엇이었는지요?
  4위. 해당 프로젝트에서 예상치 못한 문제가 발생했다면, 어떻게 대처했는지와 그 과정에서 어떤 트레이드오프를 고려했는지 말씀해 주세요.
  5위. 해당 프로젝트 진행 중 예상치 못한 변수나 문제가 발생했을 때, 어떻게 대처했으며 어떤 트레이드오프를 고려했는지 구체적으로 이야기해 주세요.
  6위. 당신이 참여했던 AI 서빙 플랫폼의 성능 향상 프로젝트에서 구체적으로 어떤 역할을 수행했는지 설명해 주세요.
  7위. 당신이 경험한 'AI 서빙 플랫폼 성능 향상' 프로젝트에서 구체적으로 어떤 문제를 해결했으며, 본인의 기여도가 어떤 방식으로 반영되었는지 설명해 주세요.
  8위. 귀하가 참여한 프로젝트 중에서 특정 기술이나 방법론을 선택한 이유와 그 선택이 프로젝트에 미친 영향에 대해 설명해 주시고, 이 과정에서의 팀원들과의 협업 경험을 함께 공유해 주실 수 있으신가요?
■ 3) AI/LLM/Agent 역량 검증
  1위. 자기소개서에서 언급한 OO 프로젝트 경험을 바탕으로, 삼성전자의 최신 AI 기술 중 하나(예: 딥러닝, 자연어 처리 등)를 어떻게 실무에 적용할 계획인지 구체적으로 설명해 주시고, 이 과정에서 예상되는 도전과제는 무엇인지 이야기해 주세요.
  2위. 해당 프로젝트에서 사용한 AI 플랫폼의 아키텍처를 선택한 이유와 그 기술이 어떻게 작동하는지 설명해 주세요.
  3